# LinkGuard — URL Safety Classifier Training

Trains the hybrid SecureBERT + URL-feature model on a free Colab T4 GPU.

**Before running:** Runtime → Change runtime type → T4 GPU

**Estimated time:** ~2 hours on T4

**Outputs saved to Google Drive** so they survive Colab disconnects.

## 1. Mount Google Drive (saves checkpoints persistently)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/linkguard_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Working dir: {DRIVE_DIR}')

## 2. Clone the repo

In [ ]:
import os

REPO_URL = 'https://github.com/vanji-creator/linkguard'
REPO_DIR = '/content/linkguard'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(f'{REPO_DIR}/linkguard-model')
print('Working directory:', os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

## 4. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## 5. Download training data

In [ ]:
# Check if we already have raw data in Drive (skip re-download)
import os, shutil

RAW_SRC  = f'{DRIVE_DIR}/raw'
RAW_DEST = 'data/raw'

os.makedirs(RAW_DEST, exist_ok=True)

if os.path.exists(RAW_SRC) and len(os.listdir(RAW_SRC)) >= 4:
    print('Raw data found in Drive — copying...')
    for f in os.listdir(RAW_SRC):
        shutil.copy(f'{RAW_SRC}/{f}', f'{RAW_DEST}/{f}')
    print('Done. Files:', os.listdir(RAW_DEST))
else:
    print('Downloading fresh data...')
    !python data/collect.py
    # Save to Drive for future runs
    os.makedirs(RAW_SRC, exist_ok=True)
    for f in os.listdir(RAW_DEST):
        shutil.copy(f'{RAW_DEST}/{f}', f'{RAW_SRC}/{f}')
    print('Raw data saved to Drive for future runs.')

## 6. Preprocess data

In [ ]:
import os

PROC_SRC  = f'{DRIVE_DIR}/processed'
PROC_DEST = 'data/processed'

os.makedirs(PROC_DEST, exist_ok=True)

if (os.path.exists(f'{PROC_SRC}/train.parquet') and
    os.path.exists(f'{PROC_SRC}/val.parquet') and
    os.path.exists(f'{PROC_SRC}/test.parquet')):
    print('Processed data found in Drive — copying...')
    import shutil
    for f in os.listdir(PROC_SRC):
        shutil.copy(f'{PROC_SRC}/{f}', f'{PROC_DEST}/{f}')
    print('Done.')
else:
    print('Preprocessing...')
    !python data/preprocess.py
    # Save to Drive
    os.makedirs(PROC_SRC, exist_ok=True)
    import shutil
    for f in os.listdir(PROC_DEST):
        shutil.copy(f'{PROC_DEST}/{f}', f'{PROC_SRC}/{f}')
    print('Processed data saved to Drive.')

## 7. Train the model

In [ ]:
# Check if a checkpoint already exists in Drive (resume training)
import os, shutil

CKPT_SRC  = f'{DRIVE_DIR}/best_model'
CKPT_DEST = 'model_output/best_model'

os.makedirs(CKPT_DEST, exist_ok=True)

if os.path.exists(f'{CKPT_SRC}/model.pt'):
    print('Checkpoint found in Drive — copying to resume training...')
    for f in os.listdir(CKPT_SRC):
        shutil.copy(f'{CKPT_SRC}/{f}', f'{CKPT_DEST}/{f}')

resume_flag = '--resume model_output/best_model/model.pt' if os.path.exists(f'{CKPT_DEST}/model.pt') else ''
print('Resume flag:', resume_flag or '(fresh training)')

In [ ]:
!python train/train.py {resume_flag}

In [ ]:
# Save checkpoint to Drive immediately after training
import shutil, os

os.makedirs(CKPT_SRC, exist_ok=True)
for f in os.listdir(CKPT_DEST):
    shutil.copy(f'{CKPT_DEST}/{f}', f'{CKPT_SRC}/{f}')
print('Checkpoint saved to Drive:', CKPT_SRC)

## 8. Evaluate

In [ ]:
!python train/evaluate.py

In [ ]:
# Show confusion matrix
from IPython.display import Image
Image('model_output/confusion_matrix.png')

In [ ]:
# Show ROC curves
from IPython.display import Image
Image('model_output/roc_curves.png')

In [ ]:
# Show threshold analysis
with open('model_output/threshold_analysis.txt') as f:
    print(f.read())

## 9. Export to ONNX

In [ ]:
!python train/export_onnx.py

## 10. Save everything to Drive

In [ ]:
import shutil, os

# ONNX models
ONNX_SRC  = 'model_output/onnx'
ONNX_DEST = f'{DRIVE_DIR}/onnx'
os.makedirs(ONNX_DEST, exist_ok=True)
for f in os.listdir(ONNX_SRC):
    shutil.copy(f'{ONNX_SRC}/{f}', f'{ONNX_DEST}/{f}')
print('ONNX models saved to Drive.')

# Evaluation outputs
EVAL_DEST = f'{DRIVE_DIR}/eval'
os.makedirs(EVAL_DEST, exist_ok=True)
for fname in ['confusion_matrix.png', 'roc_curves.png',
              'threshold_analysis.txt', 'classification_report.txt',
              'training_log.csv']:
    src = f'model_output/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{EVAL_DEST}/{fname}')
print('Eval outputs saved to Drive.')

print(f'\nAll files saved to: {DRIVE_DIR}')
print('Drive contents:', os.listdir(DRIVE_DIR))

## 11. Quick inference test

Make sure the model works before deploying.

In [ ]:
!python serve/local_inference.py

## Done!

Your trained model is in Google Drive at:
- `linkguard_training/onnx/model_int8.onnx` — deploy this
- `linkguard_training/best_model/` — tokenizer + checkpoint

**Next steps:**
1. Download `onnx/` and `best_model/` from Drive to your machine
2. Place them in `linkguard-model/model_output/`
3. Deploy `serve/spaces/` to HuggingFace Spaces
4. Set `LG_MODEL_URL` in `background.js`